<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_MobileNetV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_MobileNetV3
Custom truncated MobileNetV3-Small style network trained from scratch on VWW.
Comparison against MobileNetV2 — same training protocol, different architecture.

**Key differences vs MobileNetV2:**
- Hard-swish activation (smoother than ReLU6)
- Squeeze-and-Excitation (SE) blocks on deeper layers
- Narrower channel widths → slightly fewer parameters

Run **Model_MobileNetV2** first if you want a direct comparison checkpoint.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VWW_MobileNetV3, VWW_MobileNetV2, count_params, model_size_mb
from utils.train   import setup_device, set_seed, evaluate, train_multi_seed, plot_history

device = setup_device(seed=41)

In [ ]:
# ── Parameter count ─────────────────────────────────────────────────
m = VWW_MobileNetV3().to(device)
total, trainable = count_params(m)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Model size      : {model_size_mb(m):.2f} MB")
del m

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints" 

In [ ]:
results, best = train_multi_seed(
    model_fn     = VWW_MobileNetV3,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63, 74, 85],
    save_dir     = SAVE_DIR,
    name_prefix  = "mobilenetv3",
    pretrained   = False,
    epochs          = 50,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 8,
)

In [ ]:
plot_history(best, title=f"MobileNetV3 (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nMobileNetV3  |  Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%  |  Best: {best['best_acc']*100:.2f}% (seed {best['seed']})")
print(f"Best checkpoint: {best['save_path']}")

## MobileNetV2 vs MobileNetV3 comparison
Run `Model_MobileNetV2.ipynb` first, then come back here to compare.

In [ ]:
# ── Compare V2 vs V3 val accuracy ───────────────────────────────────
V2_CKPT = f"{SAVE_DIR}/mobilenetv2_seed_85.pth"   # adjust seed if needed

if os.path.exists(V2_CKPT):
    v2 = VWW_MobileNetV2().to(device)
    v2.load_state_dict(torch.load(V2_CKPT, map_location=device))
    v2_acc = evaluate(v2, val_loader, device)

    v3 = VWW_MobileNetV3().to(device)
    v3.load_state_dict(torch.load(best["save_path"], map_location=device))
    v3_acc = evaluate(v3, val_loader, device)

    v2_params, _ = count_params(v2)
    v3_params, _ = count_params(v3)

    print("=" * 50)
    print(f"MobileNetV2  val acc: {v2_acc*100:.2f}%  params: {v2_params:,}")
    print(f"MobileNetV3  val acc: {v3_acc*100:.2f}%  params: {v3_params:,}")
    print(f"Accuracy delta      : {(v3_acc - v2_acc)*100:+.2f}%")
    print(f"Param delta         : {v3_params - v2_params:+,}")
    print("=" * 50)
else:
    print("⚠️  Run Model_MobileNetV2.ipynb first to get the V2 checkpoint")